In [1]:
from datetime import datetime
import hashlib
import hmac
from pathlib import Path
import time
from urllib.parse import urljoin, urlsplit
from urllib.robotparser import RobotFileParser

from bs4 import BeautifulSoup
import pandas as pd
import requests

RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

INTERIM = RAIZ / "datos" / "interim"
RAW = RAIZ / "datos" / "raw" / "san_silvestre_2025"
RAW.mkdir(parents=True, exist_ok=True)

URL = "https://sansilvestrecoruna.com/gl/web/resultado/competicion-16683"
FECHA_PRUEBA = pd.Timestamp("2025-12-31")
AGENTE = "P2CorunaCorre"
sesion = requests.Session()
sesion.headers["User-Agent"] = f"{AGENTE}/1.0 (practica academica)"

robots_url = urljoin(URL, "/robots.txt")
respuesta = sesion.get(robots_url, timeout=20)
robots = RobotFileParser(robots_url)

if respuesta.status_code == 404:
    robots.parse([])
else:
    respuesta.raise_for_status()
    robots.parse(respuesta.text.splitlines())

pausa = max(1.5, robots.crawl_delay(AGENTE) or 0)

def leer_pagina(url, numero):
    sitio, destino = urlsplit(URL), urlsplit(url)
    if (destino.scheme, destino.netloc, destino.path) != (
        sitio.scheme, sitio.netloc, sitio.path
    ):
        raise RuntimeError("El enlace siguiente sale de la clasificación.")
    if not robots.can_fetch(AGENTE, url):
        raise RuntimeError("robots.txt restringe el acceso a esta página.")

    ruta = RAW / f"pagina_{numero:03d}.html"
    if not ruta.exists():
        time.sleep(pausa)
        respuesta = sesion.get(url, timeout=25)
        respuesta.raise_for_status()
        ruta.write_bytes(respuesta.content)

    return BeautifulSoup(ruta.read_bytes(), "html.parser")

print("Fuente:", URL)
print("Fecha de la prueba:", FECHA_PRUEBA.date())
print("HTTP robots.txt:", respuesta.status_code)
print("Pausa entre peticiones:", pausa, "segundos")

primera = leer_pagina(URL, 1)
print("Primera página cargada. Cabeceras:")
print([th.get_text(" ", strip=True) for th in primera.select("table th")])

Fuente: https://sansilvestrecoruna.com/gl/web/resultado/competicion-16683
Fecha de la prueba: 2025-12-31
HTTP robots.txt: 404
Pausa entre peticiones: 1.5 segundos
Primera página cargada. Cabeceras:
['', 'Puesto', 'Dorsal', 'Nombre', 'Apellidos', 'P. Sexo', 'P. Categoria', 'Tempo']


In [2]:
def extraer_filas(pagina):
    for tabla in pagina.find_all("table"):
        cabeceras = [th.get_text(" ", strip=True).casefold() for th in tabla.select("th")]
        if {"puesto", "dorsal", "nombre", "apellidos"}.issubset(cabeceras):
            break
    else:
        raise RuntimeError("No se encontró la tabla de clasificación.")

    filas = []
    for tr in tabla.select("tr"):
        celdas = tr.find_all("td", recursive=False)
        if not celdas:
            continue
        if len(celdas) - len(cabeceras) not in (0, 1) or len(celdas) < 7:
            raise RuntimeError("Ha cambiado el formato de la tabla.")

        puesto, dorsal, nombre, apellidos, sexo, categoria, tiempo = [
            celda.get_text(" ", strip=True) for celda in celdas[-7:]
        ]
        filas.append({
            "dorsal": dorsal,
            "nombre_completo": f"{nombre} {apellidos}".strip(),
            "puesto_sexo": sexo,
            "puesto_categoria": categoria,
        })

    if not filas:
        raise RuntimeError("La tabla no contiene filas.")
    return filas

filas = []
visitadas = set()
dorsales_vistos = set()
url = URL
numero = 1

while url:
    if url in visitadas:
        raise RuntimeError("La paginación repitió una URL.")
    visitadas.add(url)

    pagina = leer_pagina(url, numero)
    nuevas = extraer_filas(pagina)
    dorsales = {fila["dorsal"] for fila in nuevas}

    if "" in dorsales or len(dorsales) != len(nuevas) or dorsales & dorsales_vistos:
        raise RuntimeError(f"Dorsales vacíos o repetidos en la página {numero}.")

    dorsales_vistos.update(dorsales)
    filas.extend(nuevas)

    siguiente = next(
        (a["href"] for a in pagina.select("a[href]")
         if a.get_text(" ", strip=True).casefold() in {"seguinte", "siguiente"}),
        None,
    )
    url = urljoin(url, siguiente) if siguiente else None
    numero += 1

san = pd.DataFrame(filas)
assert len(visitadas) > 1, "Solo se encontró una página. Revisa la navegación."

san.to_csv(INTERIM / "san_silvestre_2025_local.csv", index=False)

print("Fecha de ejecución:", datetime.now().astimezone().isoformat(timespec="seconds"))
print("Páginas recorridas:", len(visitadas))
print("Personas en la clasificación:", len(san))

Fecha de ejecución: 2026-11-20T01:46:33+01:00
Páginas recorridas: 215
Personas en la clasificación: 5366


In [3]:
def normalizar_nombre(columna):
    return (
        columna.fillna("")
        .str.normalize("NFKD")
        .str.replace(r"[\u0300-\u036f]", "", regex=True)
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

variables = dict(
    linea.split("=", 1)
    for linea in (RAIZ / ".env").read_text(encoding="utf-8").splitlines()
    if "=" in linea and not linea.lstrip().startswith("#")
)
clave = variables.get("CORUNA_HMAC_KEY", "").strip().encode("utf-8")
assert clave, "Falta CORUNA_HMAC_KEY en .env."

def clave_nombre(columna):
    nombres = normalizar_nombre(columna)
    return nombres.map(
        lambda nombre: hmac.new(
            clave, f"nombre:{nombre}".encode("utf-8"), hashlib.sha256
        ).hexdigest() if nombre else ""
    )

enlace = pd.read_csv(INTERIM / "enlace_personas_local.csv", dtype="string")
inscritos = pd.read_csv(INTERIM / "inscritos_circuito_interim.csv", dtype="string")
calendario = pd.read_csv(RAIZ / "datos" / "clean" / "carreiras.csv")
calendario["fecha"] = pd.to_datetime(calendario["fecha"], errors="raise")
carreras_posteriores = calendario.loc[
    calendario["fecha"].gt(FECHA_PRUEBA), "id_carrera"
]
assert not carreras_posteriores.empty, "No hay carreras posteriores en el calendario."

enlace = enlace.loc[enlace["id_carrera"].isin(carreras_posteriores)].copy()
enlace["clave_nombre"] = clave_nombre(enlace["nombre_normalizado"])
inscritos["clave_nombre"] = clave_nombre(inscritos["nombre_completo"])

cruce_san = san.copy()
cruce_san["clave_nombre"] = clave_nombre(cruce_san["nombre_completo"])

identidades = enlace.groupby("clave_nombre")["persoa_id"].nunique()
dudosos_enlace = enlace.loc[
    ~enlace["resultado_cruce"].isin([
        "coinciden nombre y dorsal", "sin coincidencia en el listado"
    ]),
    "clave_nombre",
]
dorsales_padron = inscritos.groupby("clave_nombre")["dorsal"].nunique()

dudosa = (
    cruce_san["clave_nombre"].eq("")
    | cruce_san["clave_nombre"].duplicated(keep=False)
    | cruce_san["clave_nombre"].isin(dudosos_enlace)
    | cruce_san["clave_nombre"].isin(identidades[identidades.gt(1)].index)
    | cruce_san["clave_nombre"].isin(dorsales_padron[dorsales_padron.gt(1)].index)
)

cruce_san["corrio_despues"] = (
    ~dudosa & cruce_san["clave_nombre"].isin(enlace["clave_nombre"])
)
cruce_san["inscrito_circuito"] = (
    ~dudosa & cruce_san["clave_nombre"].isin(inscritos["clave_nombre"])
)

cruce_san["sexo"] = (
    cruce_san["puesto_sexo"].str.extract(r"^([MF])\s*-", expand=False)
    .fillna("Desconocido")
)
cruce_san["categoria"] = (
    cruce_san["puesto_categoria"]
    .str.extract(r"^([A-Z0-9]+)[MF]\s*-\s*\d+$", expand=False)
    .fillna("Desconocida")
)

print("Coincidencias por nombre con carreras posteriores:", int(cruce_san["corrio_despues"].sum()))
print("Coincidencias por nombre con inscritos:", int(cruce_san["inscrito_circuito"].sum()))
print("Registros ambiguos o sin nombre, excluidos del cruce:", int(dudosa.sum()))

for grupo in ["corrio_despues", "inscrito_circuito"]:
    print("\nDistribución por sexo y categoría de San Silvestre:", grupo)
    print(
        cruce_san.loc[cruce_san[grupo]]
        .groupby(["sexo", "categoria"]).size().rename("personas").to_string()
    )

Coincidencias por nombre con carreras posteriores: 697
Coincidencias por nombre con inscritos: 527
Registros ambiguos o sin nombre, excluidos del cruce: 41

Distribución por sexo y categoría de San Silvestre: corrio_despues
sexo  categoria
F     JV2            7
      SN            96
      VTA           59
      VTB           65
      VTC           14
      VTD            2
M     DVIS           1
      JV2            8
      SN           138
      VTA          115
      VTB          109
      VTC           64
      VTD           19

Distribución por sexo y categoría de San Silvestre: inscrito_circuito
sexo  categoria
F     JV1          13
      JV2           2
      SN           38
      VTA          41
      VTB          42
      VTC          10
      VTD           2
M     DVIS          1
      JV1          20
      JV2          12
      SN           97
      VTA          96
      VTB          86
      VTC          51
      VTD          16
